# Reward and state ablation

Compares DQN reward/state variants under the controlled replica study configuration.
The reusable training and state-variant functions live in the DQN package; this notebook orchestrates and visualizes their outputs.


# Reward and State Ablations

This notebook launches fresh DQN reward/state ablations on `matlab_env_python_replica` and renders the saved study summaries inline.

Default scenario for this notebook:
- `30 s` episode duration
- `10 s` skin-to-fat switch
- midpoint reset
- stroke limit handled with mechanical clamp mode
- force input `F_h(t) = 15 + 5 sin(5 t)`
- FE mode: `switched_dynamics`
- reward study includes the equal-gradient variant `eqgrad_t40_tr40_nojerk`


## Ablation Run Config

In [1]:
import subprocess
import sys
from pathlib import Path

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    for root in (candidate, candidate / 'TeleopWithRL'):
        if (root / 'matlab_env_python_replica').exists() and (root / 'notebooks' / '_teleop_nb.py').exists():
            for path_to_add in (root.parent, root):
                if str(path_to_add) not in sys.path:
                    sys.path.insert(0, str(path_to_add))
            break
    else:
        continue
    break
else:
    raise RuntimeError('Could not find TeleopWithRL notebook root.')

from notebooks._teleop_nb import load_csv_rows, load_json, repo_paths, run_notebook_command, show_rows

P = repo_paths()
REPO = P['repo']
WORKSPACE = REPO.parent
DQN_RESULTS = P['dqn_results']
FE_MODE_DIR = {
    'switched_dynamics': 'dyn',
    'gui_skin_locked': 'gui',
}

CFG = {
    'study_name': 'abl_30s10s_5A15B_w5_clp_01',
    'stage': 'dqn_state',
    'env_mode': 'changing_skin_fat',
    'fe_mode': 'switched_dynamics',
    'episode_duration_s': 30.0,
    'env_switch_time_s': 10.0,
    'reset_position_mode': 'midpoint',
    'stroke_limit_mode': 'clamp',
    'force_amp_N': 5.0,
    'force_bias_N': 15.0,
    'force_freq_rad_s': 5.0,
    'force_phase_rad': 0.0,
    'force_waveform': 'sine',
    'reward_variant': 'baseline_cfg',
    'dqn_episodes': 2500,
    'dqn_parallel_envs': 8,
    'test_episodes': 100,
    'seed': 42,
    'parallel_workers': 1,
    'worker_torch_threads': 1,
    'skip_existing': True,
    'resume': False,
    'disable_stroke_limit': False,
}

SUITE_ROOT = DQN_RESULTS / FE_MODE_DIR[CFG['fe_mode']] / CFG['study_name']
STAGE_ROOTS = {
    'baseline': SUITE_ROOT / '00b',
    'reward': SUITE_ROOT / '30dr',
    'state': SUITE_ROOT / '40ds',
}

CMD = [
    sys.executable,
    '-m',
    'TeleopWithRL.matlab_env_python_replica.dqn.scripts.run_experiments',
    '--study-name', CFG['study_name'],
    '--stage', CFG['stage'],
    '--env-mode', CFG['env_mode'],
    '--fe-mode', CFG['fe_mode'],
    '--episode-duration', str(CFG['episode_duration_s']),
    '--env-switch-time', str(CFG['env_switch_time_s']),
    '--reset-position-mode', CFG['reset_position_mode'],
    '--stroke-limit-mode', CFG['stroke_limit_mode'],
    '--force-amp', str(CFG['force_amp_N']),
    '--force-bias', str(CFG['force_bias_N']),
    '--force-freq-rad', str(CFG['force_freq_rad_s']),
    '--force-phase', str(CFG['force_phase_rad']),
    '--force-waveform', CFG['force_waveform'],
    '--reward-variant', CFG['reward_variant'],
    '--dqn-episodes', str(CFG['dqn_episodes']),
    '--dqn-parallel-envs', str(CFG['dqn_parallel_envs']),
    '--test-episodes', str(CFG['test_episodes']),
    '--seed', str(CFG['seed']),
    '--parallel-workers', str(CFG['parallel_workers']),
    '--worker-torch-threads', str(CFG['worker_torch_threads']),
]
if CFG['skip_existing']:
    CMD.append('--skip-existing')
if CFG['resume']:
    CMD.append('--resume')
if CFG['disable_stroke_limit']:
    CMD.append('--disable-stroke-limit')

show_rows(
    [
        {'item': 'workspace_root', 'value': str(WORKSPACE)},
        {'item': 'suite_root', 'value': str(SUITE_ROOT)},
        {'item': 'study_name', 'value': CFG['study_name']},
        {'item': 'stage', 'value': CFG['stage']},
        {'item': 'fe_mode', 'value': CFG['fe_mode']},
        {'item': 'episode_duration_s', 'value': CFG['episode_duration_s']},
        {'item': 'env_switch_time_s', 'value': CFG['env_switch_time_s']},
        {'item': 'reset_position_mode', 'value': CFG['reset_position_mode']},
        {'item': 'stroke_limit_mode', 'value': CFG['stroke_limit_mode']},
        {'item': 'force_amp_N', 'value': CFG['force_amp_N']},
        {'item': 'force_bias_N', 'value': CFG['force_bias_N']},
        {'item': 'force_freq_rad_s', 'value': CFG['force_freq_rad_s']},
        {'item': 'reward_variant', 'value': CFG['reward_variant']},
        {'item': 'dqn_episodes', 'value': CFG['dqn_episodes']},
        {'item': 'test_episodes', 'value': CFG['test_episodes']},
        {'item': 'disable_stroke_limit', 'value': CFG['disable_stroke_limit']},
        {'item': 'command', 'value': subprocess.list2cmdline(CMD)},
    ],
    title='Reward/state ablation config',
    max_rows=20,
)


### Reward/state ablation config

item,value
workspace_root,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop
suite_root,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01
study_name,abl_30s10s_5A15B_w5_clp_01
stage,dqn_state
fe_mode,switched_dynamics
episode_duration_s,30.0
env_switch_time_s,10.0
reset_position_mode,midpoint
stroke_limit_mode,clamp
force_amp_N,5.0


In [2]:
print(subprocess.list2cmdline(CMD))
completed = run_notebook_command(CMD, cwd=WORKSPACE)
print(f'Completed with return code {completed.returncode}.')


"c:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\.venv\Scripts\python.exe" -m TeleopWithRL.matlab_env_python_replica.dqn.scripts.run_experiments --study-name abl_30s10s_5A15B_w5_clp_01 --stage dqn_state --env-mode changing_skin_fat --fe-mode switched_dynamics --episode-duration 30.0 --env-switch-time 10.0 --reset-position-mode midpoint --stroke-limit-mode clamp --force-amp 5.0 --force-bias 15.0 --force-freq-rad 5.0 --force-phase 0.0 --force-waveform sine --reward-variant baseline_cfg --dqn-episodes 2500 --dqn-parallel-envs 8 --test-episodes 100 --seed 42 --parallel-workers 1 --worker-torch-threads 1 --skip-existing
Completed with return code 0.


## Artifact Locations

In [3]:
show_rows(
    [
        {
            'stage': stage_name,
            'stage_root': str(stage_root),
            'manifest_json': str(stage_root / 'study_manifest.json'),
            'summary_csv': str(stage_root / 'study_summary.csv'),
        }
        for stage_name, stage_root in STAGE_ROOTS.items()
    ],
    title='Ablation artifact paths',
    max_rows=10,
)


### Ablation artifact paths

stage,stage_root,manifest_json,summary_csv
baseline,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\00b,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\00b\study_manifest.json,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\00b\study_summary.csv
reward,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\study_manifest.json,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\study_summary.csv
state,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\study_manifest.json,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\study_summary.csv


## Baseline Summary

In [4]:
baseline_csv = STAGE_ROOTS['baseline'] / 'study_summary.csv'
baseline_json = STAGE_ROOTS['baseline'] / 'dqn' / 'l' / 'summary.json'
baseline_rows = load_csv_rows(baseline_csv) if baseline_csv.exists() else []
baseline_detail = [load_json(baseline_json)] if baseline_json.exists() else []

show_rows(baseline_rows, title='Baseline stage summary', max_rows=10)
show_rows(
    [
        {
            'label': row.get('label'),
            'reward_variant': row.get('reward_variant'),
            'tracking_rmse_m': row.get('tracking_rmse_m'),
            'transparency_rmse_w': row.get('transparency_rmse_w'),
            'invalid_episode_rate': row.get('invalid_episode_rate'),
            'stroke_limit_mode': row.get('stroke_limit_mode'),
            'reset_options': row.get('reset_options'),
        }
        for row in baseline_detail
    ],
    title='Baseline detailed summary',
    max_rows=10,
)


### Baseline stage summary

agent,study_family,variant_name,state_variant,reward_variant,tracking_rmse_m,transparency_rmse_w,pre_switch_tracking_rmse_m,post_switch_tracking_rmse_m,pre_switch_transparency_rmse_w,post_switch_transparency_rmse_w,mean_reward,invalid_episode_rate,model_path,out_dir
dqn,baselines,DQN_baseline_baseline_cfg_30s_10s,S0_baseline_full10,baseline_cfg,0.004269481950627491,1.9680466347020096,0.006352012435992468,0.0026774396718879418,2.969437982010581,1.1836512873474698,-133.65168988563556,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\00b\dqn\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\00b\dqn


### Baseline detailed summary

label,reward_variant,tracking_rmse_m,transparency_rmse_w,invalid_episode_rate,stroke_limit_mode,reset_options
DQN_baseline_baseline_cfg_30s_10s,baseline_cfg,0.004269481950627491,1.9680466347020096,0.0,clamp,"{'force_amp': 5.0, 'force_bias': 15.0, 'force_phase': 0.0, 'force_waveform': 'sine', 'fe_mode': 'switched_dynamics', 'reset_position_mode': 'midpoint', 'stroke_limit_mode': 'clamp', 'force_freq_rad': 5.0}"


## Reward Ablation Summary

In [5]:
reward_csv = STAGE_ROOTS['reward'] / 'study_summary.csv'
reward_manifest = STAGE_ROOTS['reward'] / 'study_manifest.json'
reward_rows = load_csv_rows(reward_csv) if reward_csv.exists() else []
reward_best = []
if reward_manifest.exists():
    reward_best = [load_json(reward_manifest).get('best', {})]

show_rows(reward_rows, title='Reward ablation rows', max_rows=30)
show_rows(reward_best, title='Reward ablation best row', max_rows=5)


### Reward ablation rows

agent,study_family,variant_name,state_variant,reward_variant,tracking_rmse_m,transparency_rmse_w,pre_switch_tracking_rmse_m,post_switch_tracking_rmse_m,pre_switch_transparency_rmse_w,post_switch_transparency_rmse_w,mean_reward,invalid_episode_rate,model_path,out_dir
dqn,dqn_reward_study,eqgrad_t40_tr40_nojerk,S0_baseline_full10,eqgrad_t40_tr40_nojerk,0.011195384709563994,2.494102792901066,0.015243451953106562,0.008474877253941483,3.581184724999946,1.7083269871569882,-1350.104867409844,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\eqgrad\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\eqgrad
dqn,dqn_reward_study,r01_t40_tr06_j005,S0_baseline_full10,r01_t40_tr06_j005,0.004391439504465677,1.9241739716598576,0.0067692106497784146,0.002452754568140869,2.863442301440305,1.205826398359661,-167.2461844184605,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r01\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r01
dqn,dqn_reward_study,r04_t60_tr08_j010,S0_baseline_full10,r04_t60_tr08_j010,0.004137929676753431,1.9466918873379981,0.006330425564756089,0.0023762468514816402,2.913483766633001,1.2000917161302123,-235.6768586112893,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r04\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r04
dqn,dqn_reward_study,r06_t70_tr10_j020,S0_baseline_full10,r06_t70_tr10_j020,0.004042418247494071,1.9881425913304431,0.006053899816408869,0.0024873412387430667,2.9884315423166545,1.2098367260240737,-321.8007903307908,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r06\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r06
dqn,dqn_reward_study,r09_t60_tr08_nojerk,S0_baseline_full10,r09_t60_tr08_nojerk,0.011764461376574855,6.8019047742848935,0.01377237179806359,0.010619073158676974,7.926998982438963,6.16280832120107,-2149.1691547119544,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r09\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r09
dqn,dqn_reward_study,r10_t70_tr10_nojerk,S0_baseline_full10,r10_t70_tr10_nojerk,0.0053647712669562505,2.0233248940485598,0.006536642900300177,0.004669829324195558,2.9954194578129387,1.2862723569263033,-318.3594495766524,0.0,C:\Users\aha173\OneDrive - America

### Reward ablation best row

agent,study_family,variant_name,state_variant,reward_variant,tracking_rmse_m,transparency_rmse_w,pre_switch_tracking_rmse_m,post_switch_tracking_rmse_m,pre_switch_transparency_rmse_w,post_switch_transparency_rmse_w,mean_reward,invalid_episode_rate,model_path,out_dir
dqn,dqn_reward_study,r01_t40_tr06_j005,S0_baseline_full10,r01_t40_tr06_j005,0.004391439504465677,1.9241739716598576,0.0067692106497784146,0.002452754568140869,2.863442301440305,1.205826398359661,-167.2461844184605,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r01\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\30dr\r01


## State Ablation Summary

In [6]:
state_csv = STAGE_ROOTS['state'] / 'study_summary.csv'
state_manifest = STAGE_ROOTS['state'] / 'study_manifest.json'
state_rows = load_csv_rows(state_csv) if state_csv.exists() else []
state_best = []
if state_manifest.exists():
    state_best = [load_json(state_manifest).get('best', {})]

show_rows(state_rows, title='State ablation rows', max_rows=30)
show_rows(state_best, title='State ablation best row', max_rows=5)


### State ablation rows

agent,study_family,variant_name,state_variant,reward_variant,tracking_rmse_m,transparency_rmse_w,pre_switch_tracking_rmse_m,post_switch_tracking_rmse_m,pre_switch_transparency_rmse_w,post_switch_transparency_rmse_w,mean_reward,invalid_episode_rate,model_path,out_dir
dqn,dqn_state_ablation,S0_baseline_full10,S0_baseline_full10,r01_t40_tr06_j005,0.005638999549759501,1.9747625826766184,0.0072268849465265995,0.004645808957341584,2.93938135233816,1.2367495784000861,-210.91662705670063,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S0\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S0
dqn,dqn_state_ablation,S1_no_mass_flow,S1_no_mass_flow,r01_t40_tr06_j005,0.005557254767818309,2.0149053681013456,0.007351563821717595,0.004393389980330169,2.907965974689052,1.3644165084866129,-218.17559710205416,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S1\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S1
dqn,dqn_state_ablation,S2_relative_mechanics,S2_relative_mechanics,r01_t40_tr06_j005,0.005267454693624295,2.025278530368752,0.007760645802940145,0.0033919473440386177,3.0208305670356617,1.2609207076539037,-208.2631173631003,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S2\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S2
dqn,dqn_state_ablation,S3_actuator_pressure_compact2,S3_actuator_pressure_compact2,r01_t40_tr06_j005,0.006618398780883093,2.093238938568656,0.00924258953172572,0.0047950050060671726,3.0488457230828,1.3873513467503147,-260.658924166983,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S3\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S3
dqn,dqn_state_ablation,S4_tube_coupling_pressure_compact2,S4_tube_coupling_pressure_compact2,r01_t40_tr06_j005,0.005668968920378615,2.2386670552590213,0.00797054917630871,0.0040547485550093375,3.2477475487260334,1.497836174728454,-247.01083708203677,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S4\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S4
dqn,dqn_state_ablation,S5_force_mechanics_minimal,S5_force_mechanics_minimal,r01_t40_tr06_j005,0.023015272571979658,14.502268771680344,0.023590355365735435,0.02272227374632515,11.211922759232255,15.894026969575316,-7526

### State ablation best row

agent,study_family,variant_name,state_variant,reward_variant,tracking_rmse_m,transparency_rmse_w,pre_switch_tracking_rmse_m,post_switch_tracking_rmse_m,pre_switch_transparency_rmse_w,post_switch_transparency_rmse_w,mean_reward,invalid_episode_rate,model_path,out_dir
dqn,dqn_state_ablation,S2_relative_mechanics,S2_relative_mechanics,r01_t40_tr06_j005,0.005267454693624295,2.025278530368752,0.007760645802940145,0.0033919473440386177,3.0208305670356617,1.2609207076539037,-208.2631173631003,0.0,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S2\m\dqn_model.pt,C:\Users\aha173\OneDrive - American University of Beirut\AGV research\spring VIPP obsidian notes\VIPP\VIPP naseem\VIPP\Teleop\TeleopWithRL\matlab_env_python_replica\dqn_experiments\results\dyn\abl_30s10s_5A15B_w5_clp_01\40ds\S2


## Selected results

Only the best recorded DQN model from each ablation stage is retained: `r06_t70_tr10_j020` for the reward stage and `S2_relative_mechanics` for the state stage.

| Model | Stage | Tracking RMSE [mm] | Pre-switch [mm] | Post-switch [mm] | Transparency RMSE [W] | Pre-switch [W] | Post-switch [W] | Invalid episodes |
|---|---|---:|---:|---:|---:|---:|---:|---:|
| `r06_t70_tr10_j020` | reward | **4.042** | 6.054 | 2.487 | 1.988 | 2.988 | 1.210 | 0.0% |
| `S2_relative_mechanics` | state | 5.267 | 7.761 | 3.392 | 2.025 | 3.021 | 1.261 | 0.0% |

The source notebook contains the executed DQN training outputs; this section promotes only the two best evaluation rows. Evaluation is shown with bar graphs only.

![DQN reward/state selected evaluation bars](../../results_index/figures/dqn_reward_state_best_evaluation_bars.png)

